# Frameworks 03 - Provider x Framework Matrix

Objetivo: probar las 20 combinaciones declaradas. Las cuatro combinaciones con python-runtime se ejecutan offline; las dieciseis externas se ejecutan solo con RUN_MATRIX_LIVE=1 y configuracion valida. Declarado, ready, passed y not-run son estados distintos.


**Lugar en el modelo:** esta matriz cruza de forma explícita cuatro Providers (dónde) con cuatro Frameworks (quién controla el loop).

**Evidencia exigida:** las 20 filas deben existir; las cuatro de `python-runtime` deben pasar offline y cualquier fila ejecutada debe bloquear el notebook si falla.

**Límite de la evidencia:** `declared`, `ready`, `not-run` y `passed` no son sinónimos. Sólo `passed` demuestra esa combinación en el ambiente actual.

## Parametros de ejecucion

RUN_MATRIX_LIVE=0 ejecuta las cuatro rutas offline. Usa 1 para intentar cruces externos configurados.

In [ ]:
import os
import agentic_systems as toolkit

api_coverage = [
    "toolkit.compatibility_report",
    "toolkit.compatibility_matrix",
    "toolkit.show",
    "toolkit.tool",
    "toolkit.runtime",
    "toolkit.framework",
    "toolkit.system",
    "AgenticSystem.agent",
    "Agent.run",
    "toolkit.human_result",
    "toolkit.RunResult",
]


In [ ]:
matrix_report = toolkit.compatibility_report()
toolkit.show(matrix_report)


In [ ]:
@toolkit.tool
def matrix_echo(value: str) -> dict:
    """Return a deterministic matrix marker."""
    return {"value": value}


In [ ]:
run_live_matrix = os.getenv("RUN_MATRIX_LIVE", "0") == "1"
matrix_cases = tuple(toolkit.compatibility_matrix())
matrix_results = []
for case in matrix_cases:
    should_run = case.provider == "python-runtime" or run_live_matrix
    if not should_run or not case.ready:
        matrix_results.append({
            **case.to_dict(),
            "execution": "not-run",
            "execution_reason": (
                case.reason if not case.ready else "RUN_MATRIX_LIVE=0"
            ),
        })
        continue
    runtime_config = toolkit.runtime(provider=case.provider)
    framework_config = toolkit.framework(case.framework)
    system = toolkit.system(runtime=runtime_config)
    agent = system.agent(
        name=f"matrix_{case.provider}_{case.framework}",
        instructions="Execute matrix_echo with the requested input.",
        tools=[matrix_echo],
        framework=framework_config,
    )
    result = agent.run({"tool": "matrix_echo", "input": {"value": "ok"}}, mode="eval")
    assert isinstance(result, toolkit.RunResult)
    assert result.engine == case.provider
    assert result.meta["framework_adapter"] == case.framework
    matrix_results.append(
        {
            **case.to_dict(),
            "execution": "passed" if result.ok else "failed",
            "result": toolkit.human_result(result),
        }
    )
assert len(matrix_cases) == len(matrix_results) == 20
python_rows = [row for row in matrix_results if row["provider"] == "python-runtime"]
failed_rows = [row for row in matrix_results if row["execution"] == "failed"]
assert len(python_rows) == 4
assert all(row["execution"] == "passed" for row in python_rows)
assert not failed_rows, failed_rows

status_counts = {
    status: sum(row["execution"] == status for row in matrix_results)
    for status in ("passed", "failed", "not-run")
}
matrix_evidence = {
    "declared": len(matrix_results),
    "ready": sum(bool(row["ready"]) for row in matrix_results),
    "status_counts": status_counts,
    "live_complete": status_counts["passed"] == len(matrix_results),
    "rows": matrix_results,
}
toolkit.show(matrix_evidence)


## Resultado e interpretacion

El reporte conserva las 20 filas y cuenta `passed`, `failed` y `not-run`. Las cuatro filas de `python-runtime` son un gate offline. Con `RUN_MATRIX_LIVE=1`, cualquier combinación lista que falle hace fallar el notebook; una combinación no configurada permanece `not-run` y no se presenta como evidencia live.